In [1]:
%pip install -U transformers sentence-transformers chromadb huggingface-hub sentencepiece pypdf python-docx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_XET"] = "1"

from pathlib import Path
import hashlib
import torch
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from pypdf import PdfReader
from docx import Document

print("Imports completed.")
print("GPU available:", torch.cuda.is_available())


Imports completed.
GPU available: True


In [3]:
from google.colab import drive
drive.mount("/content/drive")
print("Google Drive mounted.")


Mounted at /content/drive
Google Drive mounted.


In [8]:
from pathlib import Path

file_path = input("Enter the file path from Google Drive: ").strip()
path = Path(file_path)

print("Selected file:", path)

if not path.is_file():
    raise FileNotFoundError(
        "File not found. Enter an existing TXT, PDF or DOCX file path."
    )

print("File size:", path.stat().st_size, "bytes")

Enter the file path from Google Drive: /content/drive/MyDrive/RAG_Demo/course_info.txt
Selected file: /content/drive/MyDrive/RAG_Demo/course_info.txt
File size: 18768 bytes


In [9]:
# /content/drive/MyDrive/RAG_Demo/course_info.txt

In [10]:
sections = []
file_type = path.suffix.lower()

if file_type == ".txt":
    sections.append({"text": path.read_text(encoding="utf-8-sig"), "section": "text file"})
elif file_type == ".pdf":
    for page_number, page in enumerate(PdfReader(str(path)).pages, start=1):
        text = page.extract_text() or ""
        if text.strip():
            sections.append({"text": text, "section": f"page {page_number}"})
        else:
            print("No readable text on page", page_number, "- scanned pages need OCR.")
elif file_type == ".docx":
    document = Document(str(path))
    parts = [paragraph.text for paragraph in document.paragraphs if paragraph.text.strip()]
    for table in document.tables:
        for row in table.rows:
            parts.append(" | ".join(cell.text for cell in row.cells))
    sections.append({"text": "\n".join(parts), "section": "document body"})
else:
    raise ValueError("Use a .txt, .pdf or .docx file.")

full_text = "\n\n".join(section["text"] for section in sections)
if not full_text.strip():
    raise ValueError("The file has no readable text. Scanned PDFs require OCR.")
print("Extracted characters:", len(full_text))
print("Text preview:")
print(full_text[:1000])


Extracted characters: 18766
Text preview:
COURSE INFORMATION — LEARNTECH TRAINING CENTRE
RAG Teaching Dataset | Version 1.0

ABOUT THIS FILE
This file describes a fictional training centre named LearnTech Training Centre. All fees, schedules, contact details, facilities and policies in this file are invented for classroom teaching and RAG testing. This is not an official prospectus for any real institute. The information in this file is the source of truth for the demonstration. When information is missing, the chatbot should say that the file does not provide it.

ABOUT THE TRAINING CENTRE
LearnTech Training Centre provides practical training in programming, web development, artificial intelligence, data analysis and office tools. LearnTech Training Centre is located in Hyderabad. Courses are available to college students, recent graduates, working professionals and beginners. Training includes explanations, demonstrations, coding exercises, assignments and project work.

PYTHON PROGR

In [11]:
chunk_size = 60
chunk_overlap = 10
chunks = []
chunk_ids = []
metadata = []

for section in sections:
    words = section["text"].split()
    for start in range(0, len(words), chunk_size - chunk_overlap):
        chunk = " ".join(words[start:start + chunk_size])
        if chunk:
            chunks.append(chunk)
            chunk_ids.append(f"chunk_{len(chunks)}")
            metadata.append({"source": path.name, "section": section["section"]})
        if start + chunk_size >= len(words):
            break

print("Total chunks:", len(chunks))
print("Chunk size:", chunk_size, "words; overlap:", chunk_overlap, "words")
for index, chunk in enumerate(chunks[:3], start=1):
    print(f"Chunk {index}: {chunk}")


Total chunks: 55
Chunk size: 60 words; overlap: 10 words
Chunk 1: COURSE INFORMATION — LEARNTECH TRAINING CENTRE RAG Teaching Dataset | Version 1.0 ABOUT THIS FILE This file describes a fictional training centre named LearnTech Training Centre. All fees, schedules, contact details, facilities and policies in this file are invented for classroom teaching and RAG testing. This is not an official prospectus for any real institute. The information in this file
Chunk 2: prospectus for any real institute. The information in this file is the source of truth for the demonstration. When information is missing, the chatbot should say that the file does not provide it. ABOUT THE TRAINING CENTRE LearnTech Training Centre provides practical training in programming, web development, artificial intelligence, data analysis and office tools. LearnTech Training Centre is located in
Chunk 3: analysis and office tools. LearnTech Training Centre is located in Hyderabad. Courses are available to college stu

In [12]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
embeddings = embedding_model.encode(chunks, normalize_embeddings=True,
                                    convert_to_numpy=True, show_progress_bar=False)
print("Embedding model loaded.")
print("Embedding matrix shape:", embeddings.shape)
print("First vector, first 10 values:", embeddings[0][:10])


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.
Embedding matrix shape: (55, 384)
First vector, first 10 values: [-0.05982181 -0.01695167 -0.00533306 -0.00738614 -0.07390203  0.00953902
 -0.03520584 -0.00500026 -0.03228087  0.03097978]


In [13]:
client = chromadb.PersistentClient(path="./file_rag_database",
                                  settings=Settings(anonymized_telemetry=False))
collection_name = "file_" + hashlib.sha256(str(path).encode()).hexdigest()[:12]
# Rebuild only this file's collection to remove outdated indexed chunks.
existing_names = [item.name for item in client.list_collections()]
if collection_name in existing_names:
    client.delete_collection(collection_name)
collection = client.create_collection(name=collection_name,
                                     metadata={"hnsw:space": "cosine"},
                                     embedding_function=None)
# Small batches also support longer files without exceeding Chroma's batch limit.
for start in range(0, len(chunks), 100):
    end = start + 100
    collection.upsert(ids=chunk_ids[start:end], documents=chunks[start:end],
                      embeddings=embeddings[start:end].tolist(), metadatas=metadata[start:end])
print("Chroma collection:", collection.name)
print("Stored chunks:", collection.count())


Chroma collection: file_85df5bd58d05
Stored chunks: 55


In [14]:
def retrieve(query, top_k=3):
    query_vector = embedding_model.encode([query], normalize_embeddings=True)
    results = collection.query(query_embeddings=query_vector.tolist(),
                               n_results=min(top_k, collection.count()),
                               include=["documents", "metadatas", "distances"])
    return results

print("Retrieval function ready: query -> query embedding -> matching Chroma chunks.")


Retrieval function ready: query -> query embedding -> matching Chroma chunks.


In [15]:
def build_prompt(query, results, tokenizer):
    # Reserve space for the question before adding context; never cut off the question.
    prefix = "Answer using only the context. If the answer is missing, say you do not know.\nContext:\n"
    suffix = "\nQuestion: " + query + "\nAnswer:"
    if len(tokenizer.encode(prefix + suffix)) > 512:
        raise ValueError("The question is too long. Please shorten it.")
    context = ""
    for passage in results["documents"][0]:
        candidate = context + passage + "\n"
        if len(tokenizer.encode(prefix + candidate + suffix)) <= 512:
            context = candidate
        else:
            # Keep as much of this passage as fits; preserve the question.
            for word in passage.split():
                candidate = context + word + " "
                if len(tokenizer.encode(prefix + candidate + suffix)) > 512:
                    break
                context = candidate
            break
    return prefix + context + suffix

print("Prompt function ready: retrieved context + input query.")
print("Prompt token limit: 512. The question is preserved.")


Prompt function ready: retrieved context + input query.
Prompt token limit: 512. The question is preserved.


In [16]:
llm_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForSeq2SeqLM.from_pretrained(llm_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
llm = llm.to(device)
llm.eval()

def answer_query(query):
    results = retrieve(query)
    prompt = build_prompt(query, results, tokenizer)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.inference_mode():
        output = llm.generate(**inputs, max_new_tokens=150, do_sample=False)
    return tokenizer.decode(output[0], skip_special_tokens=True)

print("LLM loaded:", llm_name)
print("Device:", device)
print("Answer function ready.")


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LLM loaded: google/flan-t5-small
Device: cuda
Answer function ready.


In [17]:
query = input("Enter your question: ").strip()
if query:
    answer = answer_query(query)
    print("\nLLM answer:")
    print(answer)
else:
    print("Enter a question and rerun this cell.")


Enter your question: what is the python course fee?

LLM answer:
27000 rupees
